In [1]:
import numpy as np
import random
from PCAClass import PCABounding, Best_Fit_CPP
import open3d as o3d
import time
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.cm as cm
#import cupy as np


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Potential Field

In [ ]:
segment=Best_Fit_CPP("plane_segments\Hyper_meshed_noise.stl")
path1=np.linspace([75,0,0],[75,400,0],51)
path2=np.linspace([125,0,0],[125,400,0],51)
path3=np.linspace([175,0,0],[175,400,0],51)
colormaskRed= (segment.points[:,0] <=50) | (segment.points[:,0]>=200)
colormaskBlue= (segment.points[:,0] >=100) & (segment.points[:,0]<=150)
segment.colors= np.full((segment.points.shape[0], 3), [0,1,0])
segment.colors[colormaskBlue]=[0,0,1]
segment.colors[colormaskRed]=[1,0,0]

test_path=o3d.geometry.PointCloud()
points_combo=np.vstack([path1,path2,path3])+np.array([0,0,10])
test_path.points=o3d.utility.Vector3dVector(points_combo)
test_path.colors=o3d.utility.Vector3dVector(np.full((points_combo.shape[0],3), [1,.6,0]))
segment.mesh.vertex_colors = o3d.utility.Vector3dVector(segment.colors)
o3d.visualization.draw_geometries([segment.mesh, test_path],mesh_show_back_face=True)

In [ ]:
def adjust_path_with_potential_field(segment, pass_points, influence_radius=30, step_size=0.1, max_adjustment=2.0):

    # Build KDTree for nearest neighbor queries
    kdtree = segment.kd_tree
    motion=[]
    # Compute gradient using local neighborhoods
    for point in pass_points:
        neighbors = kdtree.query_ball_point(point, r=influence_radius) # Get nearest neighbors
        neighbor_colors=segment.colors[neighbors]
        red_neighbors= neighbor_colors[:,0]== 1
        blue_neighbors= neighbor_colors[:,2]== 1


        if np.any(red_neighbors):
            red_dist=segment.points[neighbors][red_neighbors]-point
            blue_dist=segment.points[neighbors][blue_neighbors]-point

            red_cum_direction=np.sum(red_dist, axis=0)
            blue_cum_direction=np.sum(blue_dist, axis=0)

            adjustment = step_size * (1 * red_cum_direction + -0.1 * blue_cum_direction)
            # Limit step magnitude without distorting direction
            norm = np.linalg.norm(adjustment)
            if norm > max_adjustment:
                adjustment = (adjustment / norm) * max_adjustment
            motion.append(adjustment)

            segment.colors[np.array(neighbors)[red_neighbors], 0] = 2
            segment.colors[np.array(neighbors)[blue_neighbors], 2] = 2

     
        else:
            motion.append(np.array([0,0,0]))
    
    return motion

passes=[path1,path2,path3]
for i in range(len(passes)):
    passes[i] += adjust_path_with_potential_field(segment, passes[i])

test_path=o3d.geometry.PointCloud()
points_combo=np.vstack([path1,path2,path3])+np.array([0,0,10])
test_path.points=o3d.utility.Vector3dVector(points_combo)
test_path.colors=o3d.utility.Vector3dVector(np.full((points_combo.shape[0],3), [1,.6,0]))
segment.mesh.vertex_colors = o3d.utility.Vector3dVector(segment.colors)
o3d.visualization.draw_geometries([segment.mesh, test_path],mesh_show_back_face=True)

#TODO: Check new path with potential field changes


## Genetic Algorithm (Not great)

In [63]:
# Genetic Algorithm (GA) Implementation
def genetic_algorithm(paths, cost_function, population_size=20, generations=50, mutation_rate=0.8, return_val=False):
    #Get population by adding random_noise to original paths
    noise_magnitude=10
    random_noise1=(np.random.randn(population_size,*paths.shape)-0.5)*noise_magnitude
    #random_noise2=(np.random.randn(population_size,*paths.shape)-0.5)*noise_magnitude
    
    direction1=np.array([1,0,0])
    #direction2=np.array([0,1,0])

    normalized_noise1=random_noise1*direction1
    #normalized_noise2=random_noise2*direction2

    #population=paths+(normalized_noise1+normalized_noise2)
    population=paths+normalized_noise1

    for gen in range(generations):

        scored_population = [(p, cost_function(p)) for p in population]
        scored_population.sort(key=lambda x: (x[1]))  # Minimize 0-time, maximize 1-time
        
        new_population = []
        for i in range(population_size // 2):
            parent1, parent2 = random.sample(scored_population[:10], 2)  # Select from top candidates
            child = crossover(parent1[0], parent2[0])
            if random.random() < mutation_rate:
                mutate(child, noise_magnitude)
            new_population.extend([child, parent1[0]])
        
        population = new_population
    best_path = min(population, key=lambda p: cost_function(p))
    
    return best_path

# Crossover function
def crossover(path1, path2):
    split = len(path1) // 2
    return np.concatenate((path1[:split], path2[split:]), axis=0) 

# Mutation function
def mutate(path, noise_magnitude=5, num_mutations=5):
    j = np.random.randint(0, path.shape[0], size=num_mutations)  # Select rows
    i = np.random.randint(0, path.shape[1], size=num_mutations)  # Select points
    mutations = (np.random.randn(num_mutations)-0.5)*noise_magnitude

    path[j, i, 0] += mutations

In [65]:
def ray_cast_prep(segment, lines):
    #Format mesh for ray-casting
    tensor_plane = o3d.t.geometry.TriangleMesh.from_legacy(segment.mesh) #OGplane= Surface to Path-Plan on
    tensor_plane.compute_triangle_normals()
    tensor_plane.compute_vertex_normals()
    scene = o3d.t.geometry.RaycastingScene()
    tensor_cast_id = scene.add_triangles(tensor_plane)
    #direction for ray-casting should be along tertiary axis, but pointing "down"
    direction=np.array(segment.tertiary_axis) * np.ones((lines[0].shape[0], 1))*-1

    return scene, direction, tensor_plane


def scan_evaluator(passes, segment, scene, direction, tensor_plane, return_val=False):
    true_colors= np.full((segment.points.shape[0], 3), (1,0,0))
    
    for point_set in passes:
        #move points "up" on z axis by max z bb height to ensure projections have intersections
        
        z_offset=segment.bounding_box.extent[-1]*segment.tertiary_axis
        #print(z_offset)
        ray_origins=point_set+z_offset
        #Ray Cast to test for intersection
        #print(ray_origins)
        #print(direction)
        LocVec=np.hstack((ray_origins.astype(dtype=np.float32), direction.astype(np.float32)))
        #print(LocVec)
        ans = scene.cast_rays(LocVec)


        #Calculate "principal vector" for interpolated
        principal_vector=point_set[0]-point_set[-1]
        #print(point_set[0],point_set[-1],principal_vector)
        principal_vector=principal_vector/np.linalg.norm(principal_vector)
        sign=np.dot(principal_vector,segment.primary_axis)
        if sign<0:
            principal_vector=-1*principal_vector

        #Determine the index of ray-casting to use for RoboDK targets
        index_hits=[p for p,q in enumerate(ans['geometry_ids']) if q==0]
        #Numpoints calculation may scale quite inappropriately, this is not refined equation
        #TODO: Test further to determine how useful numpoints actually is 
        #numpoints=5+int(segment.PCA_eigenvals[-1]//10)
        #Set first robot pose in roughly 10 mm from edge
        #RoboIndex=np.linspace(index_hits[10],index_hits[-10],numpoints).round().astype(int)
        color_updates = np.full((segment.points.shape[0],3), -1.0)

        for i in index_hits:
            dist=ans['t_hit'].numpy()[i]
            delta=dist*segment.tertiary_axis*(-1)
            onSurface=LocVec[i][:3]+delta
            intersection_index=ans['primitive_ids'].numpy()[i]
            #print(f"Intersected triangle index: {intersection_index}")
            average_normal = segment.compute_average_normal_t(tensor_plane, intersection_index)
            #print(f"Average normal of all neighbors: {average_normal}")

            #Creates RoboDK points; Uses principal_vector from above instead of primary axis due to interpolation between edges
            transformation_matrix=np.eye(3)
            transformation_matrix[0:3,0]=principal_vector
            transformation_matrix[0:3,1]=np.cross(average_normal, principal_vector)
            transformation_matrix[0:3,2]=average_normal
            #print(transformation_matrix)

            candidate_indices = np.array(segment.kd_tree.query_ball_point(onSurface, 32))
            if len(candidate_indices)>0:
                inv_rotation_matrix = np.linalg.inv(transformation_matrix)
                #print(candidate_indices)
                new_candidate_points= inv_rotation_matrix@segment.points[candidate_indices].T
                new_seed_point=inv_rotation_matrix@onSurface
                probe_mask = ((new_seed_point[0] - 10 <= new_candidate_points[0]) & (new_candidate_points[0] <= new_seed_point[0] + 10) &
                            (new_seed_point[1] - 32 <= new_candidate_points[1]) & (new_candidate_points[1] <= new_seed_point[1] + 32))
                
                scanned_points=candidate_indices[probe_mask]
                color_updates[scanned_points] = [0, 1, 0]

        update_mask = color_updates[:, 0] != -1
        already_marked_mask = (true_colors[:, 0] != 1) 
        rescanned_mask = update_mask & already_marked_mask
        true_colors[update_mask] = color_updates[update_mask]
        true_colors[rescanned_mask]=[0,0,1]
    
    #new_path=o3d.geometry.PointCloud()
    #new_path.points=o3d.utility.Vector3dVector(np.vstack(passes))
    #segment.mesh.vertex_colors = o3d.utility.Vector3dVector(true_colors)
    #o3d.visualization.draw_geometries([segment.mesh, new_path],mesh_show_back_face=True)
   
    # Compute Euclidean distances and sum along the second axis
    diffs = passes[:, 1:, :] - passes[:, :-1, :]
    total_distances = np.sum(np.linalg.norm(diffs, axis=2), axis=1)
    cost=((np.sum(total_distances)-1200)/10)**2

    matches_Red = np.sum(np.all(true_colors == np.array([1,0,0]), axis=1))
    matches_Blue=np.sum(np.all(true_colors== np.array([0,0,1]), axis=1))
    matches_Green=segment.colors.shape[0]-(matches_Blue+matches_Red)
    totalsum=matches_Blue+matches_Green+matches_Red
    cost+= (matches_Red/totalsum)*15 + (matches_Blue/totalsum)*2
    if return_val:
        return true_colors
    else:
        return cost
    
def visualizer(colors, passes):
    new_path=o3d.geometry.PointCloud()
    new_path.points=o3d.utility.Vector3dVector(np.vstack(passes))
    segment.mesh.vertex_colors = o3d.utility.Vector3dVector(colors)
    o3d.visualization.draw_geometries([segment.mesh, new_path],mesh_show_back_face=True)  
    return  

segment=Best_Fit_CPP("plane_segments\Hyper_meshed_noise.stl")
path1=np.linspace([75,0,0],[75,400,0],51)
path2=np.linspace([125,0,0],[125,400,0],51)
path3=np.linspace([175,0,0],[175,400,0],51)
colors1 = np.full((51, 3), (0, 1, 0), dtype=np.float32)  
colors2 = np.full((51, 3), (0, 0.5, 0.5))  
colors3 = np.full((51, 3), (0, 0, 1)) 

lines=np.array([path1,path2,path3])
colors=[colors1,colors2,colors3]
adjusted_lines = np.array([line + np.array([0, 0, 10]) for line in lines])

ray_scene, ray_direction, tensor_plane=ray_cast_prep(segment, adjusted_lines)
#Path visualization
trial=o3d.geometry.PointCloud()
trial.points=o3d.utility.Vector3dVector(np.vstack(adjusted_lines))
trial.colors=o3d.utility.Vector3dVector(np.vstack(colors))
#o3d.visualization.draw_geometries([segment.mesh,segment.bounding_box, trial,],mesh_show_back_face=True)


#s=time.time()
#base_score=scan_evaluator(lines, segment,ray_scene, ray_direction, tensor_plane)
#e=time.time()
#print(f" unoptimized={e-s}")
#print(base_score)


cost_function= lambda input: scan_evaluator(input, segment=segment, scene=ray_scene, direction=ray_direction, tensor_plane=tensor_plane)
best_path=genetic_algorithm(adjusted_lines, cost_function)
color=scan_evaluator(best_path, segment=segment, scene=ray_scene, direction=ray_direction, tensor_plane=tensor_plane,return_val=True)
visualizer(color,best_path)
#s2=time.time()
#e2=time.time()
#print(f"total time for checking: {e2-s2}")



-1.027020057184309
